# Customer Churn — EDA, Cleaning, and Model TrainingCompanion notebook to the [churn-analyst-agent repository](https://github.com/khizer-kt/churn-pred).It covers the three things the assessment asks a notebook to show: **what was wrong with thedata and how it was handled**, **exploratory analysis**, and **model training with a justifiedmetric choice**.One rule throughout: this notebook **imports the project's real code** rather thanre-implementing it. If cleaning were written twice, the notebook and the deployed agent wouldeventually disagree about the same dataset, and the numbers here would stop describing thesystem that actually runs.

## 0. SetupClones the repository and installs dependencies when running on Colab; does nothing when run locally from the repo root.

In [ ]:
import osimport subprocessimport sysIN_COLAB = "google.colab" in sys.modulesREPO = "https://github.com/khizer-kt/churn-pred.git"if IN_COLAB:    if not os.path.exists("churn-pred"):        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)    os.chdir("churn-pred")    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],                   check=True)sys.path.insert(0, os.getcwd())print("Working directory:", os.getcwd())

In [ ]:
import matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom src import configfrom src.data.loader import (data_quality_audit, describe_cleaning,                             load_clean, load_raw)pd.set_option("display.width", 120)plt.rcParams["figure.figsize"] = (9, 4)plt.rcParams["axes.grid"] = Trueplt.rcParams["grid.alpha"] = 0.3

## 1. The raw data, and what is wrong with itThe brief did not say what the problems were. These were found by profiling every column.

In [ ]:
raw = load_raw()print("shape:", raw.shape)raw.head()

### 1.1 `TotalCharges` is a text column with 11 disguised nullsThis is the one that hides from a routine check. The blanks are a **single space character**,so `isnull()` reports nothing at all and `astype(float)` raises.

In [ ]:
coerced = pd.to_numeric(raw["TotalCharges"], errors="coerce")print("dtype as loaded          :", raw["TotalCharges"].dtype)print("nulls reported by isnull():", raw["TotalCharges"].isnull().sum())print("actually non-numeric     :", coerced.isna().sum())print("the offending values     :", raw.loc[coerced.isna(), "TotalCharges"].unique())

In [ ]:
# Every affected row has tenure == 0 -- customers who signed up but have not been# billed a cycle. So the correct fill is 0, not the median: mean or median# imputation would invent a billing history that never happened.affected = raw.loc[coerced.isna()]print("rows affected      :", len(affected))print("all have tenure 0  :", bool((affected["tenure"] == 0).all()))print("their churn labels :", affected["Churn"].value_counts().to_dict())affected[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

### 1.2 Seven columns carry perfectly collinear sentinel levels`MultipleLines` has `"No phone service"`; six add-on columns have `"No internet service"`.Both are fully determined by another column. Left in place, one-hot encoding emits sevenduplicate dummy columns and destabilises the coefficients that per-customer attributiondepends on.

In [ ]:
violations = {}for col in config.INTERNET_ADDON_COLS:    mismatch = (raw[col] == "No internet service") != (raw["InternetService"] == "No")    violations[col] = int(mismatch.sum())mismatch = (raw["MultipleLines"] == "No phone service") != (raw["PhoneService"] == "No")violations["MultipleLines"] = int(mismatch.sum())print("rows where the sentinel disagrees with its parent column:")for col, n in violations.items():    print(f"  {col:<20} {n}")print("\n-> implication is exact, not approximate: the level carries no information of its own")

### 1.3 Forty-two rows carry contradictory labels — and are keptGroups of rows identical across all 19 features but disagreeing on `Churn`. This is**irreducible (Bayes) error, not corruption**: two customers with identical observableattributes genuinely made different decisions.They are counted and left in. Removing label noise teaches a model a certainty the world doesnot support and inflates the apparent score. It also sets a ceiling — a model reporting nearperfect separation here is leaking, not learning.

In [ ]:
audit = data_quality_audit()for k, v in audit.items():    print(f"{k:<32} {v}")feat = [c for c in raw.columns if c not in ("customerID", "Churn")]grouped = raw.groupby(feat, dropna=False)["Churn"].nunique()example = grouped[grouped > 1].index[0]raw[(raw[feat] == pd.Series(example, index=feat)).all(axis=1)][    ["customerID", "tenure", "Contract", "MonthlyCharges", "Churn"]]

### 1.4 The columns the brief's example questions ask about do not existThe sample questions mention **region**, **revenue trend** and **product category**. None arepresent, and there is **no date column at all** — the dataset is a single cross-sectionalsnapshot, so no metric can be tracked over time.This is treated as a deliberate hallucination trap: the agent publishes these absences in itsschema and refuses such questions by name rather than producing a plausible-looking breakdown.

In [ ]:
print("columns present:", list(raw.columns))for concept in ["region", "date", "revenue", "product"]:    hits = [c for c in raw.columns if concept in c.lower()]    print(f"  anything matching {concept!r}: {hits or 'none'}")

## 2. Cleaning decisionsEach fix is a separately named, separately tested function in `src/data/cleaning.py`. Theloader applies them in order and asserts the result.

In [ ]:
df = load_clean()for step in describe_cleaning():    print(f"[{step['code']}] {step['description']}")    for key, value in step.items():        if key not in ("code", "description"):            print(f"      {key}: {value}")    print()

In [ ]:
# The assertions the loader enforces on every load -- cheap insurance against a# silently changed input file.assert len(df) == 7043assert df["customerID"].is_uniqueassert df["TotalCharges"].notna().all() and df["TotalCharges"].dtype.kind == "f"assert set(df["Churn"].unique()) == {0, 1}assert (df.loc[df["tenure"] == 0, "TotalCharges"] == 0).all()print("cleaned shape:", df.shape)print("all post-load assertions passed")

## 3. Exploratory analysis

In [ ]:
churn_rate = df["Churn"].mean()print(f"overall churn rate: {churn_rate:.2%}  ({int(df['Churn'].sum())} of {len(df)})")print(f"a do-nothing classifier would score {1 - churn_rate:.2%} accuracy")print("\n-> this is why accuracy is disqualified as the headline metric")

In [ ]:
def churn_by(column, ax):    g = df.groupby(column, observed=True)["Churn"].agg(["mean", "size"]).sort_values("mean")    (g["mean"] * 100).plot(kind="barh", ax=ax, color="#4C78A8")    ax.axvline(churn_rate * 100, color="crimson", ls="--", lw=1, label="overall")    ax.set_xlabel("churn rate (%)"); ax.set_ylabel("")    ax.set_title(column); ax.legend(fontsize=8)    return gfig, axes = plt.subplots(2, 2, figsize=(13, 8))for ax, col in zip(axes.ravel(), ["Contract", "InternetService", "PaymentMethod", "TechSupport"]):    churn_by(col, ax)plt.tight_layout(); plt.show()

In [ ]:
# The single widest split in the dataset.df.groupby("Contract", observed=True)["Churn"].agg(    customers="size", churned="sum", churn_rate="mean").assign(churn_rate=lambda d: (d["churn_rate"] * 100).round(2)).sort_values("churn_rate")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))bucket = df.groupby("tenure_bucket", observed=True)["Churn"].agg(["mean", "size"])order = [b[2] for b in config.TENURE_BUCKETS if b[2] in bucket.index](bucket.loc[order, "mean"] * 100).plot(kind="bar", ax=axes[0], color="#4C78A8", rot=30)axes[0].axhline(churn_rate * 100, color="crimson", ls="--", lw=1)axes[0].set_title("churn rate by tenure cohort"); axes[0].set_ylabel("churn rate (%)")for label, sub in df.groupby("Churn"):    axes[1].hist(sub["MonthlyCharges"], bins=40, alpha=0.6,                 label="churned" if label else "retained")axes[1].set_title("MonthlyCharges is bimodal"); axes[1].set_xlabel("MonthlyCharges")axes[1].legend()plt.tight_layout(); plt.show()print("Two clusters: phone-only customers near $20, and fiber customers $70-$110.")print("Churn concentrates in the expensive cluster.")

In [ ]:
# gender is inert. Worth showing explicitly -- if the model ever surfaces it as a# top factor, something is wrong with the pipeline.for col in ["gender", "Partner", "Dependents", "SeniorCitizen"]:    g = df.groupby(col, observed=True)["Churn"].agg(["mean", "size"])    spread = (g["mean"].max() - g["mean"].min()) * 100    print(f"{col:<15} spread {spread:5.2f} pp   " +          "  ".join(f"{i}={v:.1%}(n={int(n)})" for i, (v, n) in g.iterrows()))

In [ ]:
numeric = config.NUMERIC_FEATURES + ["Churn"]corr = df[numeric].corr()fig, ax = plt.subplots(figsize=(6, 5))im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)ax.set_xticks(range(len(numeric))); ax.set_xticklabels(numeric, rotation=45, ha="right")ax.set_yticks(range(len(numeric))); ax.set_yticklabels(numeric)for i in range(len(numeric)):    for j in range(len(numeric)):        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)plt.colorbar(im); plt.title("numeric correlations"); plt.tight_layout(); plt.show()print("tenure is the strongest single numeric signal (-0.35).")print("TotalCharges correlates 0.83 with tenure -- overlapping information, kept for")print("the billing history it carries beyond tenure x MonthlyCharges.")

## 4. Training and metric selection`src/model/train.py` trains four candidates, selects between them, and persists the pipeline.Running it here produces exactly the artifacts the app and the agent load — same code, same`random_state`, same numbers.

In [ ]:
from src.model.train import main as train_modelmetrics = train_model()

In [ ]:
rows = []for name, res in metrics["models"].items():    cv = res["cv"]    rows.append({"model": name, "PR-AUC": cv["pr_auc"], "ROC-AUC": cv["roc_auc"],                 "Brier": cv["brier"], "max cal. gap": cv.get("max_calibration_gap"),                 "recall": cv["recall"]})pd.DataFrame(rows).set_index("model").sort_values("PR-AUC", ascending=False)

### Why PR-AUC is the primary metric**Accuracy is disqualified.** A do-nothing classifier scores 73.46%.**Ranking matters more than any fixed threshold.** The agent's real questions — *who is mostlikely to churn*, *which segment is riskiest* — are ordering problems, so the headline metricmust be threshold-free. Unlike ROC-AUC, PR-AUC is not flattered by the 73% negative class: itsbaseline is the churn rate itself, so the lift is honest.**Calibration is a correctness requirement, not a nicety.** This model is not a decisionsystem. It is a tool an LLM calls, whose output the agent *states to a user as a number*. Amodel that says 65% for a group that churns 33% of the time reports a figure that is computedbut wrong — which arrives with an audit trail that makes it look verified.So PR-AUC selects, and calibration gates: among candidates tied on ranking, best-calibrated wins.

In [ ]:
# The finding that changed the design: class_weight="balanced" barely moves ranking# and badly damages calibration.lr = metrics["models"]["logistic_regression"]["cv"]bal = metrics["models"]["logistic_regression_balanced"]["cv"]print(f"{'':<22}{'unweighted':>12}{'balanced':>12}")for key in ["pr_auc", "roc_auc", "brier", "max_calibration_gap"]:    print(f"{key:<22}{lr[key]:>12.4f}{bal[key]:>12.4f}")print("\nRanking is unaffected. Brier and the calibration gap are much worse when balanced,")print("because reweighting inflates every predicted probability by roughly 1/base_rate.")

In [ ]:
chosen = metrics["chosen_model"]cal = pd.DataFrame(metrics["models"][chosen]["calibration"])fig, ax = plt.subplots(figsize=(5.5, 5.5))ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")ax.plot(cal["mean_predicted"], cal["observed_rate"], "o-", color="#4C78A8", label=chosen)for _, r in cal.iterrows():    ax.annotate(f"n={int(r['n'])}", (r["mean_predicted"], r["observed_rate"]),                fontsize=7, xytext=(4, -8), textcoords="offset points")ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed churn rate")ax.set_title("Reliability curve (held-out test set)"); ax.legend()plt.tight_layout(); plt.show()cal

In [ ]:
# The decision threshold is chosen by expected cost, not left at 0.5.##   COST_FN_OVER_FP = (CLV_MARGIN x OFFER_SUCCESS_RATE) / OFFER_COST## The offer-success term is the one usually omitted: catching a churner is only# worth what the intervention actually recovers. Leaving it out inflates the# ratio to 10:1 and drives the threshold to 0.18 -- flagging 68% of all customers.print(f"cost ratio in use : {config.COST_FN_OVER_FP}")print(f"threshold chosen  : {metrics['threshold']}")print("\nsensitivity -- the assumption is load-bearing, so it is stated rather than buried:")for ratio, thr in metrics["models"][chosen]["threshold_sensitivity"].items():    print(f"  {ratio:<18} -> threshold {thr}")test = metrics["models"][chosen]["test"]print("\nheld-out test set:")for key in ["pr_auc", "roc_auc", "brier", "recall", "precision", "f2"]:    print(f"  {key:<10} {test[key]}")print(" ", test["confusion_matrix"])

## 5. The model as a callable toolThe point of Stage 1 is that the model does not stay in this notebook. The same functions theagent calls are importable and usable here.

In [ ]:
from src.model import serviceresult = service.predict_churn_risk("3668-QPYBK")print(f"risk {result['risk_score']:.1%}  ({result['risk_band']}, "      f"percentile {result['percentile']:.0f} of all customers)\n")for f in result["top_factors"]:    print(f"  {f['feature']:<18} = {str(f['value']):<16} {f['direction']:<10} "          f"{f['contribution']:+.3f}")    if f["note"]:        print(f"      {f['note']}")

In [ ]:
# Contributions are an EXACT decomposition, not an approximation: they sum to the# predicted logit. That is why the agent can state them to a user as claims.import mathpipeline, meta, _ = service._load_artifacts()row = service.score_all_customers().query("customerID == '3668-QPYBK'").iloc[0]frame = pd.DataFrame([row[config.MODEL_FEATURES].to_dict()])x = np.asarray(pipeline.named_steps["prep"].transform(frame))[0]coefs, means = np.asarray(meta["coefficients"]), np.asarray(meta["training_means"])reconstructed = float((coefs * (x - means)).sum() + coefs @ means + meta["intercept"])actual = math.log(row["risk_score"] / (1 - row["risk_score"]))print(f"sum of contributions + reference : {reconstructed:.10f}")print(f"logit of the predicted score     : {actual:.10f}")print(f"difference                       : {abs(reconstructed - actual):.2e}")

In [ ]:
# What-if: the same customer on a two-year contract.whatif = service.predict_churn_risk("3668-QPYBK", overrides={"Contract": "Two year"})print(f"today            : {whatif['baseline_risk_score']:.1%}")print(f"two-year contract: {whatif['risk_score']:.1%}")print(f"change           : {whatif['risk_delta'] * 100:+.1f} percentage points")

In [ ]:
# Aggregate risk across a segment. Predicted and observed are returned together,# which gives a calibration check on every segment query for free.seg = service.predict_segment_risk({    "Contract": "Month-to-month",    "InternetService": "Fiber optic",    "PaymentMethod": "Electronic check",})print(f"customers        : {seg['n_customers']}")print(f"mean predicted   : {seg['mean_risk']:.1%}")print(f"actually churned : {seg['actual_churn_rate']:.1%}")print(f"lift over base   : {seg['lift']}x")print(f"\npredicted and observed agree to "      f"{abs(seg['mean_risk'] - seg['actual_churn_rate']):.3f} -- the calibration work paying off")

## 6. Summary**Data issues found:** `TotalCharges` stored as text with 11 blanks invisible to `isnull()`(all `tenure == 0`, so filled with 0 rather than the median); seven perfectly collinearsentinel levels collapsed; `SeniorCitizen` recoded to match every other flag; 42contradictory-label rows and 22 near-duplicates counted and deliberately kept; and no`region`, date or product-category column, which the agent refuses by name rather thaninventing.**Metric:** PR-AUC as the primary, because the agent's questions are ranking problems andPR-AUC's baseline is the churn rate rather than the 73% negative class. Calibration gates thechoice, because the agent quotes these probabilities to users as numbers. Recall at acost-selected threshold reports the operational picture.**Model:** unweighted logistic regression — test PR-AUC 0.636 against a 0.265 no-skillbaseline, well calibrated, and the only candidate giving an exact per-customer attribution.The agent, the restricted execution tool and the numeric-grounding validator live in therepository; see the README for how planning and verification work.